# Agri Vision: continue the three alternative detectors toward round 30

Use the **current account and existing Agri Vision Drive**. YOLO is excluded. Choose T4 and Run all to train **EfficientDet → SSDLite → SSD–MobileNetV3 sequentially**. YOLO may run at the same time on its other account.

Upload `agrivision_30_round_code.zip` to `MyDrive/Agri Vision/`; the existing dataset parts remain usable. This notebook creates separate linked `_to30` result folders and preserves original runs. It checks the original code, model specifications, dataset, GPU, package versions and checkpoint hashes before continuing.

EfficientDet resumes from its latest committed round (last verified: 2, so next 3). SSDLite and SSD resume from round 10, next 11. Original cosine learning rates are preserved through round 10; rounds 11–30 use the original terminal learning rate (10% of original initial LR), with the same per-client optimizer reset. This is a documented continuation, not a fresh thirty-round schedule.


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/Agri Vision")
WORKSPACE = Path("/content/agrivision_30_round")
EXPECTED_DATASET_SHA256 = "7a2920e4a4e0405ee157e9b40ac6306b54d6933b20a306b44a4b7c6871f20868"
EXPECTED_CODE_SHA256 = "a1fe67c876bb3a2e5da123e9f5820652e91189f8fe3b21cead34381c5422f579"
TARGET_ROUNDS = 30
SEED = 42
WORKERS = 2
STAMP = "20261009_224152_449800"
MODELS = ["efficientdet_d0", "ssdlite", "ssd_mobilenetv3"]
SOURCE_ROOT = DRIVE_ROOT / "comparison_results"
SOURCES = {model: SOURCE_ROOT / f"{model}_non_chili_seed42_{STAMP}" for model in MODELS}
RUNS = {model: SOURCE_ROOT / f"{model}_non_chili_seed42_{STAMP}_to30" for model in MODELS}


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import hashlib, json, shutil, zipfile
def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream,"sha256").hexdigest()
WORKSPACE.mkdir(parents=True, exist_ok=True)
def upload_path(filename):
    path = DRIVE_ROOT / filename
    if path.is_file() or filename != "non_chili_v1_colab_dataset.zip":
        return path
    parts_dir = DRIVE_ROOT / "non_chili_v1_upload_parts"
    manifest_path = parts_dir / "parts.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
        assert manifest["joined_sha256"] == EXPECTED_DATASET_SHA256, "Wrong dataset parts manifest"
        specs = manifest["parts"]
        assert specs and [item["name"] for item in specs] == [filename + f".part{index:02d}" for index in range(1, len(specs) + 1)], "Missing or incorrectly ordered parts manifest"
        parts = [parts_dir / item["name"] for item in specs]
        assert all(part.is_file() and part.stat().st_size == item["bytes"] for part, item in zip(parts, specs)), "Upload all listed dataset parts with their complete sizes first"
    else:
        parts = [DRIVE_ROOT / (filename + suffix) for suffix in (".part01", ".part02")]
        assert all(part.is_file() for part in parts), "Upload the dataset ZIP or the dataset parts folder first"
    joined = WORKSPACE.parent / filename
    if joined.is_file() and sha(joined) == EXPECTED_DATASET_SHA256:
        return joined
    temporary = joined.with_suffix(joined.suffix + ".partial")
    with temporary.open("wb") as output:
        for part in parts:
            with part.open("rb") as source:
                shutil.copyfileobj(source, output, length=16 * 1024 * 1024)
    assert sha(temporary) == EXPECTED_DATASET_SHA256, "Dataset parts are damaged or do not belong together"
    temporary.replace(joined)
    print(f"Joined {len(parts)} dataset parts on Colab's local disk.")
    return joined
for filename, expected in [("non_chili_v1_colab_dataset.zip", EXPECTED_DATASET_SHA256),
                           ("agrivision_30_round_code.zip", EXPECTED_CODE_SHA256)]:
    path = upload_path(filename)
    assert path.is_file(), f"Upload {filename} first"
    assert sha(path) == expected, f"Wrong or damaged upload: {filename}"
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            assert (WORKSPACE / name).resolve().is_relative_to(WORKSPACE.resolve()), "Unsafe archive path"
        archive.extractall(WORKSPACE)
print("Verified and extracted both uploads.")


In [ ]:
import os, subprocess, sys
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
# Protect Colab's CUDA-enabled torch/torchvision pair from dependency replacement.
import torch, torchvision
torch_before, vision_before = torch.__version__, torchvision.__version__
assert torch.cuda.is_available(), "Select Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_name(0)
print("GPU:", gpu, "torch:", torch_before, "torchvision:", vision_before)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-r", "requirements-comparison.txt"], check=True)
# Optional model dependencies, pinned where tested. --no-deps above preserves core CUDA packages.
subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "safetensors", "packaging", "tqdm", "omegaconf==2.4.0",
                "polars>=0.20", "ultralytics-thop>=2.0.18", "matplotlib", "opencv-python", "requests", "scipy", "psutil"], check=True)
assert torch.__version__ == torch_before and torchvision.__version__ == vision_before
for pattern in ["test_comparison.py", "test_30_round_workflow.py"]:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"], check=True)


## Validate all sources before starting any model
This catches missing checkpoints or mismatched environments before spending training time. GPU access may still end during a run; completed rounds remain resumable. The other two models start after the preceding model finishes. Rerunning skips already completed continuations.


In [ ]:
from scripts.continue_comparison import validate_source
for model in MODELS:
    settings, latest, protocol = validate_source(WORKSPACE, SOURCES[model], torch.device("cuda:0"))
    assert settings["config"]["seed"] == SEED and settings["config"]["local_epochs"] == 1
    assert settings["config"]["model"] == model
    print(model, "source committed round:", latest["round"], "next:", latest["round"] + 1)
print("All source preflight checks passed.")


In [ ]:
for model in MODELS:
    print(f"Starting/resuming {model} toward round {TARGET_ROUNDS}", flush=True)
    subprocess.run([sys.executable, "-m", "scripts.continue_comparison", "--root", str(WORKSPACE),
                    "--source", str(SOURCES[model]), "--output", str(RUNS[model]),
                    "--rounds", str(TARGET_ROUNDS), "--workers", str(WORKERS)], check=True)
    print(model, json.loads((RUNS[model] / "latest.json").read_text()))


## Results, resume and storage
Each `_to30` folder records original lineage, schedule, pooled/client metrics and initial/best/latest checkpoints. Source metrics through its committed round are copied for a complete curve. Source folders are not modified. Intermediate weights in the new continuation folder are pruned only after a completed round is committed; validation metrics remain. All-round raw prediction exports are omitted to conserve Drive space.

An interruption repeats the incomplete round. Reopen this notebook on the current account, select T4 and Run all; the first incomplete model resumes and completed models finish without retraining. Do not change the source folders, upload bundle or package versions during a continuation.

These are continued ten-round experiments. The thesis must report their schedules separately from the fresh native-YOLO thirty-round experiment. Final held-out evaluation is deferred until configuration/checkpoint selection.
